In [4]:
from __future__ import annotations

import csv
import os
import numpy as np
from scipy.optimize import minimize
import pandas as pd
import matplotlib.pyplot as plt
from pypolar import mueller

from scipy.optimize import least_squares  # type: ignore[import]


In [ ]:
import numpy as np
import pandas as pd

def load_chi_from_csv(real_csv_path, imag_csv_path):

    chi_real = pd.read_csv(real_csv_path, index_col=0).to_numpy(dtype=float)
    chi_imag = pd.read_csv(imag_csv_path, index_col=0).to_numpy(dtype=float)

    chi = chi_real + 1j * chi_imag

    return chi

In [ ]:
real_csv = "Reference_20260507_chi_real.csv"
imag_csv = "Reference_20260507_chi_imag.csv"
aa = load_chi_from_csv(real_csv, imag_csv)

In [ ]:
STATE_ORDER = ["H", "V", "D", "A", "R", "L"]
PAULI_ORDER = ["I", "X", "Y", "Z"]


def ket(label: str):
    """Return a polarization ket in the H/V computational basis.

    Convention used here:
        |R> = (|H> - i |V>) / sqrt(2)
        |L> = (|H> + i |V>) / sqrt(2)

    Swap these two definitions if your optical convention is opposite.
    """
    label = label.upper()
    if label == "H":
        return np.array([1.0, 0.0], dtype=complex)
    if label == "V":
        return np.array([0.0, 1.0], dtype=complex)
    if label == "D":
        return np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
    if label == "A":
        return np.array([1.0, -1.0], dtype=complex) / np.sqrt(2)
    if label == "R":
        return np.array([1.0, -1.0j], dtype=complex) / np.sqrt(2)
    if label == "L":
        return np.array([1.0, 1.0j], dtype=complex) / np.sqrt(2)
    raise ValueError(f"Unknown polarization label: {label!r}")


def projector(label: str):
    v = ket(label)
    return np.outer(v, v.conj())


def projector_list(states=STATE_ORDER):
    return [projector(s) for s in states]


def bell_state(name = "Psi+"):
        """Return Bell state in basis [HH, HV, VH, VV].
    
        The first qubit is the reference arm; the second qubit is the channel arm.
        """
        H = ket("H")
        V = ket("V")
        HH = np.kron(H, H)
        HV = np.kron(H, V)
        VH = np.kron(V, H)
        VV = np.kron(V, V)
        
        name = name.replace(" ", "").lower()
    
        if name in ["phi+", "phiplus", "phi_plus"]:
    
            state = (HH + VV) / np.sqrt(2)
    
        elif name in ["phi-", "phiminus", "phi_minus"]:
    
            state = (HH - VV) / np.sqrt(2)
    
        elif name in ["psi+", "psiplus", "psi_plus"]:
    
            state = (HV + VH) / np.sqrt(2)
    
        elif name in ["psi-", "psiminus", "psi_minus"]:
    
            state = (HV - VH) / np.sqrt(2)
    
        else:
    
            raise ValueError(
    
                "Unknown Bell state. Use 'Phi+', 'Phi-', 'Psi+', or 'Psi-'."
            )
            
        return np.outer(state, state.conj())

############################################################
############################################################

def _params_to_complex_matrix(params: np.ndarray, rows: int = 8, cols: int = 2):
    """Convert real vector to a complex matrix A of shape (rows, cols)."""
    params = np.asarray(params, dtype=float)
    need = 2 * rows * cols
    if params.size != need:
        raise ValueError(f"Expected {need} real parameters, got {params.size}.")
    real = params[: rows * cols].reshape(rows, cols)
    imag = params[rows * cols :].reshape(rows, cols)
    return real + 1j * imag


def _complex_matrix_to_params(A: np.ndarray):
    A = np.asarray(A, dtype=complex)
    return np.concatenate([A.real.ravel(), A.imag.ravel()])


def params_to_kraus(params: np.ndarray, n_kraus: int = 4):
    """Map unconstrained real parameters to CPTP Kraus operators.

    A complex 2*n_kraus by 2 matrix is orthonormalized to an isometry V using
    QR decomposition.  V is then split into n_kraus blocks of size 2x2.

    Since V^dagger V = I, the resulting Kraus operators obey
        sum_mu K_mu^dagger K_mu = I.
    """
    if n_kraus < 1:
        raise ValueError("n_kraus must be >= 1.")
    rows = 2 * n_kraus
    A = _params_to_complex_matrix(params, rows=rows, cols=2)

    # Avoid a rank-deficient starting point.  The tiny diagonal bias is harmless
    # during optimization but stabilizes QR if all parameters are accidentally zero.
    A = A.copy()
    A[0, 0] += 1e-12
    A[1, 1] += 1e-12

    Q, R = np.linalg.qr(A, mode="reduced")

    # Fix arbitrary QR column phases for a more continuous parameterization.
    diag = np.diag(R)
    phase = np.ones_like(diag, dtype=complex)
    nz = np.abs(diag) > 1e-14
    phase[nz] = diag[nz] / np.abs(diag[nz])
    Q = Q * phase.conj()[None, :]

    kraus = [Q[2 * mu : 2 * (mu + 1), :] for mu in range(n_kraus)]
    return kraus


def kraus_tp_error(kraus: list[np.ndarray]):
    accum = np.zeros((2, 2), dtype=complex)
    for K in kraus:
        accum += K.conj().T @ K
    return accum - np.eye(2, dtype=complex)


def initial_identity_params(n_kraus: int = 4, noise: float = 1e-3, seed: int | None = 1):
    """Initial parameters near the identity channel."""
    rng = np.random.default_rng(seed)
    V = np.zeros((2 * n_kraus, 2), dtype=complex)
    V[0:2, :] = np.eye(2, dtype=complex)  # K0 = I, others = 0
    V = V + noise * (rng.normal(size=V.shape) + 1j * rng.normal(size=V.shape))
    return _complex_matrix_to_params(V)

############################################################
############################################################



In [ ]:
def third_col_to_matrix(input_csv, output_csv):
    df = pd.read_csv(input_csv, header=None)
    data = pd.to_numeric(df.iloc[:, 2], errors="raise").to_numpy()

    if len(data) != 36:
        raise ValueError("The third column must have exactly 36 values.")

    matrix = data.reshape(6, 6)
    pd.DataFrame(matrix).to_csv(output_csv, index=False, header=False)
    return matrix

In [ ]:
Input_csv = "Arb_PMF1m_01_20260507"
Output_csv = Input_csv + "_matrix"
aaa = third_col_to_matrix(Input_csv + ".csv", Output_csv + ".csv")

In [ ]:
aa = np.loadtxt(Output_csv, delimiter=",")
print(aa)

In [ ]:
def ZeroClip(x):
    y = x * 1e8
    y = np.round(y)
    y /= 1e8
    return y

In [ ]:
def convert_Mueller_raw(input_csv, output_csv):
    label_map = {1: "H", 2: "V", 3: "D", 4: "A", 5: "R", 6: "L"}

    df = pd.read_csv(input_csv, header=None)
    df.columns = ["Signal", "Idler", "count"]

    df["Signal"] = df["Signal"].astype(int).map(label_map)
    df["Idler"] = df["Idler"].astype(int).map(label_map)
    df["count"] = df["count"].astype(int)

    df.to_csv(output_csv, index=False)
    return df

In [ ]:
def print_complex_matrix(M: np.ndarray, name: str = "matrix", precision: int = 4) -> None:
    """Print a compact complex matrix."""
    M = np.asarray(M)
    print(f"{name} =")
    for row in M:
        entries = []
        for z in row:
            entries.append(f"{z.real:+.{precision}f}{z.imag:+.{precision}f}j")
        print("  [" + ", ".join(entries) + "]")

In [ ]:
Folder_name = "Folder_name_test"
os.makedirs(Folder_name, exist_ok=True)
aa = convert_Mueller_raw("Arb_PMF1m_02_20260507.csv", f"{Folder_name}/Arb_PMF1m_02_20260507_convert.csv")

In [2]:
Rotation_angle = np.pi / 20
Rotation = np.identity(4)
Rotation[1:3, 1:3] = [[np.cos(2 * Rotation_angle), np.sin(2 * Rotation_angle)],
                      [-np.sin(2 * Rotation_angle), np.cos(2 * Rotation_angle)]]

QWP_retardance = np.pi / 2
QWP = np.identity(4)
QWP[2:4, 2:4] = [[np.cos(QWP_retardance), np.sin(QWP_retardance)],
                   [-np.sin(QWP_retardance), np.cos(QWP_retardance)]]

HWP_retardance = np.pi
HWP = np.identity(4)
HWP[2:4, 2:4] = [[np.cos(HWP_retardance), np.sin(HWP_retardance)],
                   [-np.sin(HWP_retardance), np.cos(HWP_retardance)]]

print("QWP")
print(Rotation.T @ QWP @ Rotation)

print("Pypolar QWP")
QWP_pypolar = mueller.op_quarter_wave_plate(Rotation_angle)
print(QWP_pypolar)

print("HWP")
print(Rotation.T @ HWP @ Rotation)

print("Pypolar HWP")
HWP_pypolar = mueller.op_half_wave_plate(Rotation_angle)
print(HWP_pypolar)

QWP
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  9.04508497e-01  2.93892626e-01 -3.09016994e-01]
 [ 0.00000000e+00  2.93892626e-01  9.54915028e-02  9.51056516e-01]
 [ 0.00000000e+00  3.09016994e-01 -9.51056516e-01  6.12323400e-17]]
Pypolar QWP
[[ 1.          0.          0.          0.        ]
 [ 0.          0.9045085   0.29389263 -0.30901699]
 [ 0.          0.29389263  0.0954915   0.95105652]
 [ 0.          0.30901699 -0.95105652  0.        ]]
HWP
[[ 1.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  8.09016994e-01  5.87785252e-01 -3.78436673e-17]
 [ 0.00000000e+00  5.87785252e-01 -8.09016994e-01  1.16470832e-16]
 [ 0.00000000e+00  3.78436673e-17 -1.16470832e-16 -1.00000000e+00]]
Pypolar HWP
[[ 1.          0.          0.          0.        ]
 [ 0.          0.80901699  0.58778525  0.        ]
 [ 0.          0.58778525 -0.80901699  0.        ]
 [ 0.          0.          0.         -1.        ]]


In [3]:
np.set_printoptions(precision = 2, suppress = True)
theta = -14
QWP_sample = mueller.op_quarter_wave_plate(np.radians(theta))
QWP_com_1 = mueller.op_quarter_wave_plate(np.radians(13.7))
QWP_com_2 = mueller.op_quarter_wave_plate(np.radians(14.2))
HWP_com = mueller.op_half_wave_plate(np.radians(74.5))
WP_com = QWP_com_2 @ HWP_com @ QWP_com_1
print("Sample")
print(QWP_sample)
print("Compensator")
print(WP_com)

total_Mueller = np.dot(QWP_sample, WP_com)
print("After compensation")
print(np.linalg.diagonal(total_Mueller))
print(total_Mueller)

Sample
[[ 1.    0.    0.    0.  ]
 [ 0.    0.78 -0.41  0.47]
 [ 0.   -0.41  0.22  0.88]
 [ 0.   -0.47 -0.88  0.  ]]
Compensator
[[ 1.    0.    0.    0.  ]
 [ 0.   -0.15 -0.61 -0.78]
 [ 0.   -0.6   0.68 -0.42]
 [ 0.    0.79  0.41 -0.47]]
After compensation
[1.   0.5  0.76 0.74]
[[ 1.    0.    0.    0.  ]
 [ 0.    0.5  -0.57 -0.65]
 [ 0.    0.62  0.76 -0.18]
 [ 0.    0.6  -0.31  0.74]]


In [5]:
HWP_com = mueller.op_half_wave_plate(np.radians(45.0))
print(HWP_com)

[[ 1.  0.  0.  0.]
 [ 0. -1.  0.  0.]
 [ 0.  0.  1.  0.]
 [ 0.  0.  0. -1.]]
